In [1]:
# ============================================================
# IMPORTS
# ============================================================

import pandas as pd
import numpy as np
import warnings
import sys
import os

warnings.filterwarnings("ignore")
sys.path.append(os.path.abspath(".."))

from functions.tools import configure_notebook_display, load_dataset

configure_notebook_display()

In [2]:
trusted_readings = load_dataset(
    "../data/trusted_readings.csv",
    date_columns=["date"])

analysis_dataset = load_dataset(
    "../data/analysis_dataset.csv",
    date_columns=["date", "sowing_date"])

Dataset loaded from: ../data/trusted_readings.csv
Dataset loaded from: ../data/analysis_dataset.csv


This analysis investigates how operational states evolve over time within each parcel.

The goal is to identify:
- stable operational behavior,
- recurring degradation patterns,

Transitions such as:
`OK -> ERROR`
or
`ERROR -> OK`


In [3]:
# ============================================
# Sensor State Transition Matrix Analysis
# ============================================

# Sorting chronologically per parcel
df_transition = trusted_readings.copy()

df_transition = df_transition.sort_values(
    by=["parcel_id", "date"])


# Previous sensor state
df_transition["previous_status"] = (
    df_transition
    .groupby("parcel_id")["sensor_status"]
    .shift(1))


df_transition["status_transition"] = (
    df_transition["previous_status"].astype(str)
    + " -> "
    + df_transition["sensor_status"].astype(str))


# Removing first observations with no previous state
transition_analysis = df_transition[
    df_transition["previous_status"].notna()
    ].copy()


transition_counts = (
    transition_analysis["status_transition"]
    .value_counts()
    .reset_index())

transition_counts.columns = ["transition", "count"]

print("\nMost Common Sensor State Transitions")
display(transition_counts.head(10))


Most Common Sensor State Transitions


,transition,count
0,OK -> OK,2658
1,ERROR -> OK,222
2,OK -> ERROR,216
3,OK -> UNKNOWN,126
4,UNKNOWN -> OK,121
5,ERROR -> ERROR,18
6,UNKNOWN -> ERROR,11
7,UNKNOWN -> UNKNOWN,5
8,ERROR -> UNKNOWN,5



The sensor ecosystem appears operationally stable overall, with the majority of consecutive observations remaining in an `OK → OK` state.

At the same time, frequent `OK → ERROR` and `ERROR → OK` transitions suggest the presence of little but recoverable sensor degradation rather than regular systemic failure. This indicates that operational instability exists locally within the ecosystem.

Additionally, `UNKNOWN` states rarely persist across consecutive observations and often transition back into operational states, suggesting that missing statuses likely reflect temporary metadata or ingestion inconsistencies rather than sustained sensor outages.

In [4]:
# ============================================
# Flatline Sensor Detection
# ============================================

# Sorting data chronologically
df_flatline = analysis_dataset.copy()

df_flatline = df_flatline.sort_values(
    by=["parcel_id", "date"])


# Detecting consecutive identical NDVI values
df_flatline["previous_ndvi"] = (
    df_flatline
    .groupby("parcel_id")["ndvi_value"]
    .shift(1))


df_flatline["is_same_as_previous"] = (
    df_flatline["ndvi_value"]
    == df_flatline["previous_ndvi"])


df_flatline["flatline_group"] = (
    df_flatline["is_same_as_previous"]
    != df_flatline["is_same_as_previous"].shift()
    ).cumsum()


flatline_sequences = (
    df_flatline[df_flatline["is_same_as_previous"]]
    .groupby(
        [
            "parcel_id",
            "ndvi_value",
            "flatline_group"
        ]
    )
    .size()
    .reset_index(name="consecutive_days"))


# Filtering suspicious flatline periods
SUSPICIOUS_THRESHOLD = 3

suspicious_flatlines = flatline_sequences[
    flatline_sequences["consecutive_days"]
    >= SUSPICIOUS_THRESHOLD
    ].sort_values(
    by="consecutive_days",
    ascending=False)


print(f"""
Total suspicious flatline sequences:
{len(suspicious_flatlines)}""")


Total suspicious flatline sequences:
0


No significant exact-value NDVI flatlining patterns were detected across consecutive observations.

This suggests that the sensing ecosystem remains dynamically responsive over time, with no strong evidence of frozen-value transmission or persistently stale NDVI reporting behavior within the observed parcels.

In [5]:
# ============================================
# Confidence-Weighted NDVI Comparison
# ============================================


df_weighted = analysis_dataset.copy()


quality_weights = {
    "TRUSTED": 1.0,
    "DEGRADED": 0.6,
    "UNKNOWN": 0.3,
    "CORRUPTED": 0.0}

df_weighted["quality_weight"] = (
    df_weighted["observation_quality"]
    .map(quality_weights))


df_weighted = df_weighted[
    df_weighted["quality_weight"] > 0
    ].copy()

# Standard Mean NDVI
standard_mean = (
    df_weighted
    .groupby("crop_type")["ndvi_value"]
    .mean()
    .reset_index(name="standard_mean_ndvi"))


# Confidence Weighted Mean NDVI
weighted_mean = (
    df_weighted
    .groupby("crop_type")
    .apply(
        lambda x: np.average(
            x["ndvi_value"],
            weights=x["quality_weight"]
        )
    )
    .reset_index(name="weighted_mean_ndvi"))


ndvi_comparison = standard_mean.merge(
    weighted_mean,
    on="crop_type")

# Calculating the difference
ndvi_comparison["difference"] = (
    ndvi_comparison["weighted_mean_ndvi"]
    - ndvi_comparison["standard_mean_ndvi"])

print("Standard vs Confidence-Weighted NDVI")
display(ndvi_comparison)

Standard vs Confidence-Weighted NDVI


,crop_type,standard_mean_ndvi,weighted_mean_ndvi,difference
0,soybean,0.398682,0.398748,0.000066
1,sugarcane,0.505071,0.504749,-0.000322
2,wheat,0.384483,0.385438,0.000955



Confidence-weighted NDVI aggregation produced only minimal deviations from the standard crop-level averages.

This suggests that while degraded and uncertain observations exist within the sensing system, they do not materially dominate the overall vegetation signal across crops. The underlying NDVI patterns therefore appear relatively robust to moderate operational uncertainty.

The preprocessing and trust-aware weighting strategy improved analytical confidence and interpretability without substantially altering the vegetation conclusions derived from the dataset. This can be significant for modeling and forecasting.